# Compare two simulations

Side-by-side comparison of two LFRic experiments from `common.EXPERIMENTS` or the AWS test runs
(`exp_a`, `exp_b` and their difference `exp_a - exp_b`), following the plots in `Cheesy-Plots.ipynb`.

In [ ]:
import warnings
from dataclasses import replace
from datetime import datetime
from functools import partial

import iris
import matplotlib.colors as mcol
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import paths
from aeolus.calc import spatial_mean, time_mean, zonal_mean
from aeolus.coord import (
    interp_cube_from_height_to_pressure_levels,
    regrid_3d,
    roll_cube_pm180,
)
from aeolus.lfric import load_lfric_raw
from aeolus.model import lfric
from aeolus.plot import figsave, tex2cf_units, unit_format, subplot_label_generator
from common import (
    DHJ_BASE,
    EXPERIMENTS,
    AWS_EXPERIMENTS,
    LATLON_GLOB,
    PA_TO_BAR,
    SHJ_BASE,
    TIME_ORIGIN,
    Diag,
    lfric_callback_uniform_height,
)

warnings.filterwarnings("ignore", category=UserWarning, module="iris")
iris.FUTURE.date_microseconds = True

## Experiments

Besides `common.EXPERIMENTS` (stored in `data/lfric/<group>/<label>/`), the AWS test runs
are loaded from `data/lfric/aws-test/pack-final-uoe-lfric/pack/<label>/output/work/<cycle>/lfric_atm/`.

In [ ]:
ALL_EXPERIMENTS = {**EXPERIMENTS, **AWS_EXPERIMENTS}
print(*ALL_EXPERIMENTS.keys(), sep="\n")

In [ ]:
exp_a = "dhj_c48_l66"
exp_b = "hd209_dt30_nrad4"
N_DAYS_MEAN = 200  # length of the averaging period
SAVE_FIGS = True

EXPS = [exp_a, exp_b]
# Average both experiments over the same days, at the end of the shorter run
# (e.g. the AWS DHJ runs only cover the first 100 days)
END_DAY = min(ALL_EXPERIMENTS[exp_key].run_length for exp_key in EXPS)
START_DAY = END_DAY - N_DAYS_MEAN
PERIOD_STR = f"days {START_DAY}-{END_DAY}"
print(PERIOD_STR)

In [ ]:
def file_end_date(fname):
    """End date of the output in a file named like `*_YYYYMMDD-YYYYMMDD.nc`."""
    return datetime.strptime(fname.stem.split("_")[-1].split("-")[1], "%Y%m%d")


def load_exp(exp_key, start_day=START_DAY, end_day=END_DAY, file_glob=LATLON_GLOB):
    """Load the lat-lon output files covering days `start_day`-`end_day` of one experiment."""
    exp = ALL_EXPERIMENTS[exp_key]
    if exp_key in AWS_EXPERIMENTS:
        fnames = sorted(
            (paths.aws_pack / exp.label / "output" / "work").glob(
                f"*/lfric_atm/{file_glob}"
            )
        )
    else:
        fnames = sorted(
            (paths.data_work / "lfric" / exp.group / exp.label).glob(file_glob)
        )
    if not fnames:
        raise FileNotFoundError(f"No {file_glob} files found for {exp_key}")
    origin = datetime.strptime(TIME_ORIGIN, "%Y-%m-%d %H:%M:%S")
    end_days = [(file_end_date(f) - origin).days for f in fnames]
    if max(end_days) < end_day:
        raise ValueError(f"{exp_key} output ends on day {max(end_days)} < {end_day}")
    fnames = [f for f, d in zip(fnames, end_days) if start_day < d <= end_day]
    # print(fnames)
    return load_lfric_raw(
        fnames,
        callback=partial(
            lfric_callback_uniform_height,
            model_top_height=exp.const.domain_height.data,
        ),
    )


def time_mean_on_pres_grid(dset):
    """Put 3D cubes on the grid of air pressure and average over all loaded times."""
    pres = dset.extract_cube("air_pressure")
    out = iris.cube.CubeList()
    for cube in dset:
        if lfric.z in [i.name() for i in cube.dim_coords]:
            cube = regrid_3d(cube, pres, model=lfric)
        out.append(time_mean(cube, model=lfric))
    return out


ds_all = {}
for exp_key in EXPS:
    ds_all[exp_key] = time_mean_on_pres_grid(load_exp(exp_key))
    print(exp_key)
    print(ds_all[exp_key])
    print()

In [ ]:
def on_grid_of_a(cube_b, cube_a):
    """Regrid `cube_b` to the grid of `cube_a` and align metadata for subtraction."""
    cube_b = regrid_3d(cube_b, cube_a, model=lfric)
    cube_b = cube_b.copy()
    cube_b.units = cube_a.units
    for coord in cube_a.dim_coords:
        if cube_b.coords(coord.name()):
            cube_b.coord(coord.name()).var_name = coord.var_name
            cube_b.coord(coord.name()).long_name = coord.long_name
            cube_b.coord(coord.name()).coord_system = coord.coord_system
    iris.util.equalise_attributes([cube_a, cube_b])
    return cube_b


def exp_title(exp_key):
    """Experiment title, or its label if both experiments have the same title."""
    if ALL_EXPERIMENTS[exp_a].title == ALL_EXPERIMENTS[exp_b].title:
        return ALL_EXPERIMENTS[exp_key].label
    return ALL_EXPERIMENTS[exp_key].title


def comparison_title():
    """Common title of the two experiments, or both titles if they differ."""
    title_a, title_b = ALL_EXPERIMENTS[exp_a].title, ALL_EXPERIMENTS[exp_b].title
    return title_a if title_a == title_b else f"{title_a} vs {title_b}"


def diff_title():
    # return f"Difference ({exp_title(exp_a)} - {exp_title(exp_b)})"
    return "Difference (b) minus (a)"


def common_kw_plt(diag, cubes):
    """Use the same colour limits in both experiment panels."""
    kw = {"extend": "both", **diag.kw_plt}
    if "levels" not in kw:
        vmin = kw.pop("vmin", min(float(c.data.min()) for c in cubes))
        vmax = kw.pop("vmax", max(float(c.data.max()) for c in cubes))
        kw["levels"] = mticker.MaxNLocator(nbins=10).tick_values(vmin, vmax)
    return kw


def make_axes(fig):
    return fig.subplot_mosaic(
        [
            ["exp_a", "exp_b", "diff"],
            ["cbar_exp_a", "cbar_exp_b", "cbar_diff"],
        ],
        gridspec_kw={"height_ratios": [1, 0.05]},
    )


def pres_profile_bar(exp_key):
    """Horizontal-mean pressure profile (bar) used as the vertical axis."""
    pres_prof = spatial_mean(ds_all[exp_key].extract_cube("air_pressure"))
    pres_prof.convert_units("Pa")
    return pres_prof.data * PA_TO_BAR


def centre_substellar(cube, exp_key):
    """Roll a cube to -180...180 deg longitude with the substellar point at 0 deg.

    The data are shifted by the experiment's `substellar_lon` (rounded to the
    nearest grid point); the longitude coordinate itself is unchanged.
    """
    dlon = np.diff(cube.coord(lfric.x).points).mean()
    shift = -round(ALL_EXPERIMENTS[exp_key].substellar_lon / dlon)
    return roll_cube_pm180(cube, add_shift=shift, model=lfric)


def exp_label_for_fname():
    return f"{exp_a}__vs__{exp_b}"

In [ ]:
DIAGS = {
    "temp": Diag(
        title="Air Temperature",
        units="$K$",
        recipe=lambda cl: cl.extract_cube("air_temperature"),
        kw_plt={"cmap": "plasma"},
    ),
    "u": Diag(
        title="Eastward Wind",
        units="$km$ $s^{-1}$",
        recipe=lambda cl: cl.extract_cube("eastward_wind"),
        kw_plt={"cmap": "inferno"},
    ),
    "w": Diag(
        title="Upward Wind",
        units="$m$ $s^{-1}$",
        recipe=lambda cl: cl.extract_cube("upward_air_velocity"),
        kw_plt={"cmap": "PuOr_r"},
    ),
}

Colour limits of the experiment panels are shared and computed from the data unless `vmin`/`vmax` or `levels` are set in `kw_plt` of a diagnostic.

## Zonal-mean plots

In [ ]:
diag_key = "u"
collapse_func = zonal_mean

fig = plt.figure(figsize=(12, 3), layout="constrained")
fig.suptitle(
    comparison_title()
    + f"\nZonal-Mean {DIAGS[diag_key].title} [{DIAGS[diag_key].units}] | {PERIOD_STR}",
    fontsize="medium",
)
axd = make_axes(fig)

two_cubes = []
for exp_key in EXPS:
    cube = collapse_func(DIAGS[diag_key].recipe(ds_all[exp_key]))
    cube.convert_units(tex2cf_units(DIAGS[diag_key].units))
    two_cubes.append(cube)
kw_plt = common_kw_plt(DIAGS[diag_key], two_cubes)

pres_lim = (np.inf, 0)
iletters = subplot_label_generator()
for ax_key, exp_key, cube in zip(["exp_a", "exp_b"], EXPS, two_cubes):
    pres_prof = pres_profile_bar(exp_key)
    pres_lim = (min(pres_lim[0], pres_prof.min()), max(pres_lim[1], pres_prof.max()))
    axd[ax_key].set_title(f"({next(iletters)})", loc="left")
    axd[ax_key].set_title(exp_title(exp_key))
    cntr = axd[ax_key].contourf(
        cube.coord("latitude").points, pres_prof, cube.data, **kw_plt
    )
    fig.colorbar(cntr, cax=axd[f"cbar_{ax_key}"], orientation="horizontal")

diff_cube = on_grid_of_a(two_cubes[1], two_cubes[0]) - two_cubes[0]
axd["diff"].set_title(f"({next(iletters)})", loc="left")
axd["diff"].set_title(diff_title())
cntr = axd["diff"].contourf(
    diff_cube.coord("latitude").points,
    pres_profile_bar(exp_a),
    diff_cube.data,
    norm=mcol.CenteredNorm(0),
    cmap="RdBu_r",
    levels=11,
    extend="both",
)
fig.colorbar(cntr, cax=axd["cbar_diff"], orientation="horizontal")

for ax in axd.values():
    if not ax.get_label().startswith("cbar_"):
        ax.set_xlabel("Latitude")
        ax.set_ylabel("Pressure (bar)")
        ax.set_yscale("log")
        ax.set_ylim(pres_lim[1], pres_lim[0])
        ax.set_xlim(-90, 90)
        ax.set_xticks(np.arange(-90, 91, 30))
        ax.grid(True, linestyle="--", alpha=0.5, dash_capstyle="round")

if SAVE_FIGS:
    figsave(fig, paths.drafts / "diffs" / f"{exp_label_for_fname()}__{diag_key}_zm")

## Longitude-latitude maps

In [ ]:
diag_key = "temp"
LEVELS = [
    1e2,
    1e3,
    1e4,
    1e5,
    1e6,
    1e7,
]  # Pa; give several levels to stack them one below another
overlay_stream = True


def to_plev(cube, exp_key, level):
    """Interpolate a time-mean cube to a pressure level, centred on the substellar point."""
    p_cube = ds_all[exp_key].extract_cube("air_pressure").copy()
    p_cube.rename(lfric.pres)
    cube_plev = interp_cube_from_height_to_pressure_levels(
        cube, p_cube, levels=[level], model=lfric
    )
    return centre_substellar(cube_plev, exp_key)


def make_axes_levels(fig, nlev):
    """One subfigure with a row of (exp_a, exp_b, diff) panels per pressure level."""
    subfigs = [fig] if nlev == 1 else fig.subfigures(nlev, 1)
    return subfigs, [make_axes(subfig) for subfig in subfigs]


LEVELS = np.atleast_1d(LEVELS)
fig = plt.figure(figsize=(12, 3 * len(LEVELS)), layout="constrained")
subfigs, axd_rows = make_axes_levels(fig, len(LEVELS))

iletters = subplot_label_generator()
for ilev, level in enumerate(LEVELS):
    axd = axd_rows[ilev]
    if len(LEVELS) > 1:
        subfigs[ilev].suptitle(
            f"Level: {unit_format(level * PA_TO_BAR, 'bar')}",
            fontsize="medium",
            fontweight="bold",
        )

    two_cubes = []
    for exp_key in EXPS:
        cube = to_plev(DIAGS[diag_key].recipe(ds_all[exp_key]), exp_key, level)
        cube.convert_units(tex2cf_units(DIAGS[diag_key].units))
        two_cubes.append(cube)
    kw_plt = common_kw_plt(DIAGS[diag_key], two_cubes)

    for ax_key, exp_key, cube in zip(["exp_a", "exp_b"], EXPS, two_cubes):
        ax = axd[ax_key]
        ax.set_title(f"({next(iletters)})", loc="left")
        if ilev == 0:
            ax.set_title(exp_title(exp_key))
        cntr = ax.contourf(
            cube.coord("longitude").points,
            cube.coord("latitude").points,
            cube.data,
            **kw_plt,
        )
        fig.colorbar(cntr, cax=axd[f"cbar_{ax_key}"], orientation="horizontal")

        if overlay_stream:
            _u = to_plev(ds_all[exp_key].extract_cube("eastward_wind"), exp_key, level)
            _v = to_plev(ds_all[exp_key].extract_cube("northward_wind"), exp_key, level)
            wspd = (_u**2 + _v**2) ** 0.5
            lons2d, lats2d = np.meshgrid(
                _u.coord(lfric.x).points, _u.coord(lfric.y).points
            )
            ax.streamplot(
                lons2d,
                lats2d,
                _u.data,
                _v.data,
                color="w",
                linewidth=wspd.data / wspd.data.max(),
                arrowstyle="wedge",
                arrowsize=0.75,
            )

    diff_cube = on_grid_of_a(two_cubes[1], two_cubes[0]) - two_cubes[0]
    ax = axd["diff"]
    ax.set_title(f"({next(iletters)})", loc="left")
    if ilev == 0:
        ax.set_title(diff_title())
    cntr = ax.contourf(
        diff_cube.coord("longitude").points,
        diff_cube.coord("latitude").points,
        diff_cube.data,
        norm=mcol.CenteredNorm(0),
        cmap="RdBu_r",
        levels=11,
        extend="both",
    )
    fig.colorbar(cntr, cax=axd["cbar_diff"], orientation="horizontal")

for ilev, axd in enumerate(axd_rows):
    for ax_key, ax in axd.items():
        if ax_key.startswith("cbar_"):
            continue
        if ilev == len(LEVELS) - 1:
            ax.set_xlabel("Longitude [deg]")
        if ax_key == "exp_a":
            ax.set_ylabel("Latitude [deg]")
        ax.set_xlim(-180, 180)
        ax.set_xticks(np.arange(-180, 181, 60))
        ax.set_ylim(-90, 90)
        ax.set_yticks(np.arange(-90, 91, 30))
        ax.grid(True, linestyle="--", alpha=0.5, dash_capstyle="round")

fig.suptitle(
    comparison_title()
    + "\n"
    + " | ".join(
        filter(
            None,
            [
                f"{DIAGS[diag_key].title} [{DIAGS[diag_key].units}]"
                + (" & Horizontal Wind Streamlines" if overlay_stream else ""),
                (
                    f"Level: {unit_format(LEVELS[0] * PA_TO_BAR, 'bar')}"
                    if len(LEVELS) == 1
                    else ""
                ),
                PERIOD_STR,
            ],
        )
    ),
    fontsize="medium",
)

if SAVE_FIGS:
    figsave(
        fig,
        paths.drafts
        / "diffs"
        / (
            f"{exp_label_for_fname()}__{diag_key}_ll_"
            + "_".join(f"{lev:.0f}pa" for lev in LEVELS)
        ),
    )

## Equatorial cross-sections

In [ ]:
diag_key = "w"
collapse_func = lambda cube: cube.interpolate(  # noqa: E731
    [("latitude", 0)], iris.analysis.Linear()
)

fig = plt.figure(figsize=(12, 3), layout="constrained")
fig.suptitle(
    comparison_title()
    + f"\nEquatorial Cross-Section of {DIAGS[diag_key].title} [{DIAGS[diag_key].units}]"
    f" | {PERIOD_STR}",
    fontsize="medium",
)
axd = make_axes(fig)

two_cubes = []
for exp_key in EXPS:
    cube = centre_substellar(
        collapse_func(DIAGS[diag_key].recipe(ds_all[exp_key])), exp_key
    )
    cube.convert_units(tex2cf_units(DIAGS[diag_key].units))
    two_cubes.append(cube)
kw_plt = common_kw_plt(DIAGS[diag_key], two_cubes)

pres_lim = (np.inf, 0)
iletters = subplot_label_generator()
for ax_key, exp_key, cube in zip(["exp_a", "exp_b"], EXPS, two_cubes):
    pres_prof = pres_profile_bar(exp_key)
    pres_lim = (min(pres_lim[0], pres_prof.min()), max(pres_lim[1], pres_prof.max()))
    axd[ax_key].set_title(f"({next(iletters)})", loc="left")
    axd[ax_key].set_title(exp_title(exp_key))
    cntr = axd[ax_key].contourf(
        cube.coord("longitude").points, pres_prof, cube.data, **kw_plt
    )
    fig.colorbar(cntr, cax=axd[f"cbar_{ax_key}"], orientation="horizontal")

diff_cube = on_grid_of_a(two_cubes[1], two_cubes[0]) - two_cubes[0]
axd["diff"].set_title(f"({next(iletters)})", loc="left")
axd["diff"].set_title(diff_title())
cntr = axd["diff"].contourf(
    diff_cube.coord("longitude").points,
    pres_profile_bar(exp_a),
    diff_cube.data,
    norm=mcol.CenteredNorm(0),
    cmap="RdBu_r",
    levels=11,
    extend="both",
)
fig.colorbar(cntr, cax=axd["cbar_diff"], orientation="horizontal")

for ax in axd.values():
    if not ax.get_label().startswith("cbar_"):
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Pressure (bar)")
        ax.set_yscale("log")
        ax.set_ylim(pres_lim[1], pres_lim[0])
        ax.set_xlim(-180, 180)
        ax.set_xticks(np.arange(-180, 181, 60))
        ax.grid(True, linestyle="--", alpha=0.5, dash_capstyle="round")

if SAVE_FIGS:
    figsave(fig, paths.drafts / "diffs" / f"{exp_label_for_fname()}__{diag_key}_eq")